In [3]:
import json
annt_json_path='/home/v-jinjzhao/datasets/jierun/ref_msra_test_coco_o365.json'
with open('output-jierun/prediction_during_eval.jsonl') as f:
    preds=[json.loads(line) for line in f]
with open(annt_json_path) as f:
    annts=json.load(f)
print(len(preds),len(annts))

45341 45341


In [4]:
from tqdm import tqdm
pred_annts=[i['ann_id'] for i in preds]
for annt in tqdm(annts):
    if annt['ann_id'] not in pred_annts:
        print(annt['ann_id'])

 13%|█▎        | 5741/45341 [00:00<00:00, 57399.11it/s]

100%|██████████| 45341/45341 [00:07<00:00, 5868.63it/s] 


In [5]:
pred_bboxes=[]
gt_bboxes=[]
for pred,annt in zip(preds,annts):
    if(len(pred['pred_bboxes'])==0):
        pred_bboxes.append([0,0,0,0])
    else:
        pred_bboxes.append(pred['pred_bboxes'][0])
    gt_bbox=annt['bbox']
    x,y,w,h=gt_bbox
    x1,y1,x2,y2=[x,y,x+w,y+h]
    gt_bboxes.append([x1,y1,x2,y2])

In [6]:
import json
import torch
from tqdm import tqdm
from overlaps import bbox_overlaps
# import average from math
from statistics import mean


def calculate_iou_acc(pred_bboxes,gt_bboxes, thresh=0.5):
    """
    pred_bboxes: [N,4]
    gt_bboxes: [N,4]
    calculate the iou and acc of the pred_bboxes and gt_bboxes,
    if iou(pred_bboxes[i],gt_bboxes[i])>0.5, then acc+=1
    all pred_bboxes_i and gt_bboxes_i are one to one assigned.
    
    """
    iou=bbox_overlaps(pred_bboxes,gt_bboxes,mode='iou', is_aligned=True)
    if(type(thresh) is not list):
        thresh=[thresh]
    accs=dict()
    for t in thresh:
        accs[t]=(iou>t).sum().item()/len(iou)
    return iou,accs




pred_bboxes=torch.tensor(pred_bboxes)
gt_bboxes=torch.tensor(gt_bboxes)
# create a list fron 0.5 to 0.9 with step 0.05
thresh=[i/100 for i in range(50,95,5)]
print(thresh)
iou,acc=calculate_iou_acc(pred_bboxes,gt_bboxes,thresh)
print_ths=[0.5,0.7,0.9]
for k,v in acc.items():
    if(k in print_ths):
        print(f"thresh={k}, acc={v}")
print(f"iou|0.5:0.9={mean([v for v in acc.values()])}")

print(iou)
print(acc)
print(f"{len(iou)}/{len(annts)}")

[0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]
thresh=0.5, acc=0.6590061974813083
thresh=0.7, acc=0.6047506671665821
thresh=0.9, acc=0.45583467501819547
iou|0.5:0.9=0.587670222437872
tensor([0.9350, 0.0418, 0.9523,  ..., 0.0000, 0.0000, 0.0000])
{0.5: 0.6590061974813083, 0.55: 0.6468538408945546, 0.6: 0.6344809333715622, 0.65: 0.6209170507928806, 0.7: 0.6047506671665821, 0.75: 0.5854745153393176, 0.8: 0.5590084029906707, 0.85: 0.5227057188857767, 0.9: 0.45583467501819547}
45341/45341


In [7]:
for annt,pred_bbox,iou_i in zip(annts,pred_bboxes,iou):
    annt['Lenna_pred']=dict(
        pred_bboxes=pred_bbox.tolist(),
        iou=iou_i.item()
    )


In [8]:
with open(annt_json_path,'w') as f:
    json.dump(annts,f,indent=4)

In [27]:
annts

[{'annt_id': '0000000',
  'bbox': [315.54, 56.12, 323.02, 384.14],
  'bbox_area': 36959.305749999985,
  'caption': 'The individual in question appears to be a woman dressed in a black long-sleeve top with a detailed lace-like design on the back. She is wearing a white skirt that ends above the knee, paired with black leggings and distinctive black shoes with pink soles. She seems to be standing indoors, possibly in a queue, and her posture suggests that she may be waiting for her turn or engaged in conversation with someone not visible in the closely cropped image. There is no visible text on her clothing, and she is not engaging with any objects or other people in a way that suggests a specific activity. Her overall attire gives a casual yet put-together look.',
  'category_id': 1,
  'file_name': '000000002685.jpg',
  'height': 555,
  'image_id': 'coco_000000002685',
  'recognition_category': '',
  'width': 640,
  'source': 'coco2017',
  'original_subset': 'val',
  'raw_caption': 'The